In [0]:
CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM `{CATALOG}`.`{SCHEMA}`.
             `pipeline_execution_logs`
        ORDER BY execution_start DESC
    """)
)

## Checking Uniqueness

In [0]:
checks = [
    ("silver_repositories", "repo_id"),
    ("silver_commits", "commit_sha"),
    ("silver_issues", "issue_id"),
    ("silver_pull_requests", "pr_id"),
]

for table, key in checks:
    df = spark.table(
        f"{CATALOG}.{SCHEMA}.{table}"
    )

    total = df.count()

    unique = (
        df.select(key)
        .distinct()
        .count()
    )

    print(
        table,
        "| total =", total,
        "| unique =", unique,
        "| PASS =", total == unique
    )

In [0]:
from pyspark.sql import functions as F
tables = [
    "bronze_repositories",
    "bronze_commits",
    "bronze_issues",
    "bronze_pull_requests",
    "silver_repositories",
    "silver_commits",
    "silver_issues",
    "silver_pull_requests",
]

for table in tables:
    null_count = (
        spark.table(
            f"{CATALOG}.{SCHEMA}.{table}"
        )
        .filter(
            F.col("load_timestamp").isNull()
        )
        .count()
    )

    print(
        table,
        "NULL load_timestamp =",
        null_count
    )

## Checking quarantine

In [0]:
display(
    spark.table(
        f"{CATALOG}.{SCHEMA}."
        "quarantine_records"
    )
)